In [4]:
import os

# make it only use GPU 0
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# If I don't do this, there are warnings
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

In [5]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPool2D, Dropout, Flatten, Dense, \
    BatchNormalization, LeakyReLU

In [12]:
n_samples, n_channels = 3000, 16
# n_samples, n_channels = 3105, 2

In [ ]:
conv2d_kwargs = {
    'use_bias': False,  # no bias because we use BatchNorm afterward
    'activation': None,  # activation is applied after BatchNorm
    'kernel_regularizer': tf.keras.regularizers.l1_l2(1e-9, 1e-9),
}
LEAKY_RELU_NEGATIVE_SLOPE = 0.3

In [13]:
def block_after_conv(layer_idx: int, pool_size: int):
    return [
        BatchNormalization(name=f'batch_norm{layer_idx}'),
        LeakyReLU(negative_slope=LEAKY_RELU_NEGATIVE_SLOPE, name=f'leaky_relu{layer_idx}'),
        MaxPool2D([pool_size, 1], padding='same', name=f'max_pool{layer_idx}'),
        Dropout(0.2, name=f'dropout{layer_idx}')
    ]

In [15]:
def conv_block1(layer_idx: int, kernel_size: int, n_filters: int, pool_size: int):
    return [
        Conv2D(n_filters, [kernel_size, 1], padding='same', **conv2d_kwargs, name=f'conv{layer_idx}'),
        *block_after_conv(layer_idx, pool_size)
    ]

In [16]:
def conv_block2(layer_idx: int, kernel_size: int, n_filters1: int, n_filters2: int, pool_size: int):
    return [
        Conv2D(n_filters1, [kernel_size, 1], padding='valid', **conv2d_kwargs, name=f'conv{layer_idx}.1'),
        Conv2D(n_filters2, [kernel_size, 1], padding='valid', **conv2d_kwargs, name=f'conv{layer_idx}.2'),
        *block_after_conv(layer_idx, pool_size)
    ]

In [17]:
model = tf.keras.models.Sequential([
    Input([n_samples, n_channels, 1]),
    BatchNormalization(name='batch_norm0'),

    *conv_block1(1, kernel_size=5, n_filters=32, pool_size=5),
    *conv_block1(2, kernel_size=5, n_filters=64, pool_size=3),
    *conv_block1(3, kernel_size=3, n_filters=96, pool_size=2),
    *conv_block1(4, kernel_size=3, n_filters=128, pool_size=2),

    *conv_block2(5, kernel_size=4, n_filters1=128, n_filters2=96, pool_size=2),
    *conv_block2(6, kernel_size=4, n_filters1=64, n_filters2=32, pool_size=2),
    *conv_block2(7, kernel_size=4, n_filters1=32, n_filters2=32, pool_size=2),

    Flatten(name='flatten8'),
    Dropout(0.5, name='dropout8'),

    Dense(64, activation=None, name='dense9'),
    LeakyReLU(negative_slope=LEAKY_RELU_NEGATIVE_SLOPE, name='leaky_relu9'),

    Dense(1, activation='sigmoid', name='output')
], name='CNN_Eberlein',
)
print(f'#layers: {len(model.layers)}')
model.summary()

#layers: 44


Model: "CNN_Eberlein"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ batch_norm0                     │ (None, 3000, 16, 1)    │             4 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 3000, 16, 32)   │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm1                     │ (None, 3000, 16, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu1 (LeakyReLU)         │ (None, 3000, 16, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool1 (MaxPooling2D)        │ (None, 600, 16, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 600, 16, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 600, 16, 64)    │        10,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm2                     │ (None, 600, 16, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu2 (LeakyReLU)         │ (None, 600, 16, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool2 (MaxPooling2D)        │ (None, 200, 16, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout2 (Dropout)              │ (None, 200, 16, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 200, 16, 96)    │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm3                     │ (None, 200, 16, 96)    │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu3 (LeakyReLU)         │ (None, 200, 16, 96)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool3 (MaxPooling2D)        │ (None, 100, 16, 96)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout3 (Dropout)              │ (None, 100, 16, 96)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4 (Conv2D)                  │ (None, 100, 16, 128)   │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm4                     │ (None, 100, 16, 128)   │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu4 (LeakyReLU)         │ (None, 100, 16, 128)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool4 (MaxPooling2D)        │ (None, 50, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout4 (Dropout)              │ (None, 50, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv5.1 (Conv2D)                │ (None, 47, 16, 128)    │        65,536 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 256,165 (1000.64 KB)

 Trainable params: 255,203 (996.89 KB)

 Non-trainable params: 962 (3.76 KB)